# Insight YOLO 训练 Notebook（高恢复性版）
本 notebook 重点优化“Kaggle 断连后恢复”：
- 统一使用 `/kaggle/working` 持久化 runs/datasets/exports
- 自动扫描并恢复最近一次数据集
- 训练支持断点续训，状态会写入恢复文件


## 1) 环境准备与恢复工具


In [ ]:
# 1. 检查 GPU
!nvidia-smi

# 2. 安装依赖
!pip install -q -U ultralytics roboflow pyyaml

# 3. 公共工具 + 持久化路径
import os, json, zipfile, glob, random, shutil, time
from pathlib import Path

WORK_ROOT = '/kaggle/working/InsightYOLO'
RUNS_PROJECT = os.path.join(WORK_ROOT, 'runs')
DATASETS_ROOT = os.path.join(WORK_ROOT, 'datasets')
EXPORTS_ROOT = os.path.join(WORK_ROOT, 'exports')
STATE_FILE = os.path.join(WORK_ROOT, 'kaggle_recovery_state.json')

os.makedirs(WORK_ROOT, exist_ok=True)
os.makedirs(RUNS_PROJECT, exist_ok=True)
os.makedirs(DATASETS_ROOT, exist_ok=True)
os.makedirs(EXPORTS_ROOT, exist_ok=True)

print(f'✅ WORK_ROOT: {WORK_ROOT}')
print(f'✅ RUNS_PROJECT: {RUNS_PROJECT}')
print(f'✅ DATASETS_ROOT: {DATASETS_ROOT}')
print(f'✅ EXPORTS_ROOT: {EXPORTS_ROOT}')
print(f'✅ STATE_FILE: {STATE_FILE}')

if not os.path.exists('/kaggle'):
    print('⚠️ 当前环境看起来不是 Kaggle，请确认运行环境。')

def _json_safe(obj):
    try:
        json.dumps(obj, ensure_ascii=False)
        return obj
    except Exception:
        return str(obj)

def _safe_tag(text):
    s = str(text or '').strip().lower()
    safe = ''.join(ch if ch.isalnum() else '_' for ch in s)
    while '__' in safe:
        safe = safe.replace('__', '_')
    return safe.strip('_')

def load_state():
    if not os.path.exists(STATE_FILE):
        return {}
    try:
        with open(STATE_FILE, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception:
        return {}

def save_state(**kwargs):
    state = load_state()
    for k, v in kwargs.items():
        state[k] = _json_safe(v)
    state['updated_at'] = time.strftime('%Y-%m-%d %H:%M:%S')
    with open(STATE_FILE, 'w', encoding='utf-8') as f:
        json.dump(state, f, ensure_ascii=False, indent=2)
    return state

def load_train_context(dataset_path, fallback_cfg=None):
    # 从 dataset_path 加载 data.yaml + training_config.json
    # 返回: dataset_path, data_yaml, train_cfg, model_family, model_size, model_name
    cfg = {
        'yolo_version': 'yolov8',
        'model_size': 'n',
        'epochs': 100,
        'batch_size': 16,
        'img_size': 640,
        'patience': 50,
        'workers': 8,
        'device': 0,
        'data_yaml': 'data.yaml'
    }
    if fallback_cfg:
        cfg.update(fallback_cfg)

    cfg_path = os.path.join(dataset_path, 'training_config.json')
    if os.path.exists(cfg_path):
        with open(cfg_path, 'r', encoding='utf-8') as f:
            file_cfg = json.load(f)
        cfg.update(file_cfg)
        print(f'✅ 已读取训练参数: {cfg_path}')
    else:
        print('⚠️ 未找到 training_config.json，将使用默认/回退参数')

    if os.path.isabs(str(cfg.get('data_yaml', 'data.yaml'))):
        data_yaml = str(cfg.get('data_yaml'))
    else:
        data_yaml = os.path.join(dataset_path, str(cfg.get('data_yaml', 'data.yaml')))

    if not os.path.exists(data_yaml):
        raise FileNotFoundError(f'data.yaml 不存在: {data_yaml}')

        model_family = str(cfg.get('yolo_version', 'yolov8'))
    # Normalize only YOLO11: yolo11 -> yolov11
    if model_family == 'yolo11':
        model_family = 'yolov11'
    supported = {'yolov8', 'yolov9', 'yolov10', 'yolov11', 'yolo26'}
    if model_family not in supported:
        print(f'⚠️ 未识别的 YOLO 版本: {model_family}（Ultralytics 可能不支持）')
    model_size = str(cfg.get('model_size', 'n')).lower()
    model_name = f'{model_family}{model_size}.pt'

    print(f'✅ 数据集路径: {dataset_path}')
    print(f'✅ data.yaml: {data_yaml}')
    print(f'✅ 模型: {model_name}')
    print('✅ 训练参数:', {k: cfg.get(k) for k in ['epochs','batch_size','img_size','patience','workers','device']})

    return dataset_path, data_yaml, cfg, model_family, model_size, model_name

def build_run_name(dataset_path, model_family, model_size):
    dataset_name = os.path.basename(os.path.normpath(str(dataset_path)))
    dataset_tag = _safe_tag(dataset_name) or 'dataset'
    return f'{model_family}{model_size}_{dataset_tag}_train', dataset_tag

def resolve_resume_checkpoint(dataset_path, model_family, model_size, runs_project, state=None):
    state = state or {}
    default_run_name, dataset_tag = build_run_name(dataset_path, model_family, model_size)
    legacy_run_name = f'{model_family}{model_size}_train'

    state_dataset = str(state.get('last_dataset_path', ''))
    state_run_name = None
    if state_dataset == str(dataset_path) and state.get('run_name'):
        state_run_name = str(state.get('run_name'))

    candidates = []

    def add_candidate(priority, reason, ckpt):
        ckpt = str(ckpt) if ckpt else ''
        if ckpt and os.path.exists(ckpt):
            candidates.append({
                'priority': int(priority),
                'reason': str(reason),
                'ckpt': ckpt,
                'ts': os.path.getmtime(ckpt)
            })

    # 优先级 0：状态文件中记录、且同数据集的 checkpoint
    state_resume_ckpt = state.get('resume_ckpt')
    if state_run_name and state_resume_ckpt:
        add_candidate(0, 'state.resume_ckpt', state_resume_ckpt)

    # 优先级 1/2/3：state run_name -> dataset 默认 run_name -> 旧版 run_name
    candidate_names = []
    for name in [state_run_name, default_run_name, legacy_run_name]:
        if name and name not in candidate_names:
            candidate_names.append(name)

    for idx, name in enumerate(candidate_names):
        ckpt = os.path.join(runs_project, name, 'weights', 'last.pt')
        add_candidate(1 + idx, f'run_name:{name}', ckpt)

    # 优先级 9：扫描同模型家族历史 run（仅兜底）
    pattern = os.path.join(runs_project, f'{model_family}{model_size}*_train', 'weights', 'last.pt')
    for ckpt in glob.glob(pattern):
        add_candidate(9, 'scan_model_family', ckpt)

    if candidates:
        dedup = {}
        for item in candidates:
            ckpt = item['ckpt']
            prev = dedup.get(ckpt)
            if prev is None:
                dedup[ckpt] = item
                continue
            if item['priority'] < prev['priority']:
                dedup[ckpt] = item
            elif item['priority'] == prev['priority'] and item['ts'] > prev['ts']:
                dedup[ckpt] = item

        best = sorted(dedup.values(), key=lambda x: (x['priority'], -x['ts']))[0]
        resume_ckpt = best['ckpt']
        resume_reason = best['reason']
        run_name = os.path.basename(os.path.dirname(os.path.dirname(resume_ckpt)))
        return run_name, resume_ckpt, True, resume_reason, dataset_tag

    run_name = state_run_name or default_run_name
    resume_ckpt = os.path.join(runs_project, run_name, 'weights', 'last.pt')
    return run_name, resume_ckpt, False, 'no_checkpoint', dataset_tag

# ---- GPU 检测与 device 规范化 ----

def _normalize_device_str(s):
    if s is None:
        return None
    s = str(s).strip()
    if not s:
        return None
    # 兼容中文逗号与空格
    s = s.replace('，', ',')
    s = s.replace(' ', '')
    return s

def _parse_device_list(s):
    s = _normalize_device_str(s)
    if not s:
        return []
    parts = [p for p in s.split(',') if p != '']
    out = []
    for p in parts:
        try:
            out.append(int(p))
        except Exception:
            pass
    return out

# 在 Kaggle 环境输出实际 GPU 数量与型号
try:
    import torch
    gpu_count = torch.cuda.device_count()
    print(f'✅ torch.cuda.device_count(): {gpu_count}')
    if gpu_count > 0:
        names = [torch.cuda.get_device_name(i) for i in range(gpu_count)]
        print('✅ GPU Names:', names)
except Exception as e:
    print('⚠️ 无法读取 torch GPU 信息:', e)

try:
    # 尝试用 nvidia-smi 进一步确认
    import subprocess
    smi = subprocess.check_output(['nvidia-smi', '-L'], text=True).strip()
    if smi:
        print('✅ nvidia-smi -L:')
        print(smi)
except Exception:
    pass


## 2) 数据源与恢复（推荐先执行 2.0）
优先执行 2.0 自动恢复；若找不到数据，再执行 2.1/2.2/2.3 其中一个。


### 2.0 自动恢复最近一次数据集（推荐）


In [ ]:
state = load_state()
state_path = state.get('last_dataset_path')
print(f'恢复状态文件: {STATE_FILE}')
if state:
    print(f'状态更新时间: {state.get("updated_at")}')

# 候选 1：状态文件记录的数据集
candidates = []
if state_path and os.path.exists(os.path.join(state_path, 'data.yaml')):
    stamp = os.path.getmtime(os.path.join(state_path, 'data.yaml'))
    candidates.append(('state', state_path, stamp))

# 候选 2：扫描 DATASETS_ROOT 下所有 data.yaml
yaml_candidates = glob.glob(os.path.join(DATASETS_ROOT, '**', 'data.yaml'), recursive=True)
for y in yaml_candidates:
    root = os.path.dirname(y)
    cfg_file = os.path.join(root, 'training_config.json')
    stamp = os.path.getmtime(cfg_file) if os.path.exists(cfg_file) else os.path.getmtime(y)
    candidates.append(('scan', root, stamp))

if not candidates:
    raise FileNotFoundError(
        f'未找到可恢复数据集。请先执行 2.1 上传 ZIP，或 2.2 指定 Kaggle 数据集。\n扫描目录: {DATASETS_ROOT}'
    )

# 去重后选最近修改
latest = {}
for src, path, ts in candidates:
    if path not in latest or ts > latest[path]['ts']:
        latest[path] = {'src': src, 'ts': ts}

best_path = sorted(latest.items(), key=lambda kv: kv[1]['ts'], reverse=True)[0][0]
meta = latest[best_path]

dataset_path, data_yaml, train_cfg, model_family, model_size, model_name = load_train_context(best_path)
save_state(
    last_dataset_path=dataset_path,
    last_source=f'auto_recover:{meta["src"]}',
    train_cfg=train_cfg,
    model_family=model_family,
    model_size=model_size,
    model_name=model_name
)

print(f'✅ 自动恢复成功: {dataset_path}')
print(f'   来源: {meta["src"]}, 时间戳: {time.ctime(meta["ts"])}')


### 2.1 Insight 导出的 ZIP（Kaggle 上传后自动识别）


In [ ]:
import os, glob, zipfile, shutil, time
from pathlib import Path

# Kaggle 使用说明：
# 1) 将 Insight 导出的 zip 上传到 Kaggle（Add Data 或放到 /kaggle/working）
# 2) 或使用下面的“手动选择 ZIP”按钮上传（会在本 Cell 内等待并自动继续）
# 3) 如果自动识别不到，可手动指定 ZIP_PATH
ZIP_PATH = None  # 例如: '/kaggle/input/insight-export/your_dataset.zip'
force_reextract = False  # 需要强制覆盖解压时改成 True

# 手动上传（按钮）
MANUAL_UPLOAD = True
UPLOAD_DIR = '/kaggle/working'
WAIT_FOR_UPLOAD = True
WAIT_TIMEOUT_SEC = 600

uploader = None
if MANUAL_UPLOAD:
    try:
        import ipywidgets as widgets
        from IPython.display import display, clear_output

        uploader = widgets.FileUpload(accept='.zip', multiple=False, description='选择 ZIP')
        status = widgets.Output()

        def _save_upload(change):
            with status:
                clear_output()
                if not uploader.value:
                    print('尚未选择文件')
                    return
                for name, info in uploader.value.items():
                    save_path = os.path.join(UPLOAD_DIR, name)
                    with open(save_path, 'wb') as f:
                        f.write(info['content'])
                    print(f'✅ 已上传: {save_path}')

        uploader.observe(_save_upload, names='value')
        display(uploader, status)
        print('请点击“选择 ZIP”上传文件，本 Cell 会等待上传完成后自动继续。')
    except Exception as e:
        print(f'⚠️ 手动上传按钮不可用: {e}')

# 等待上传完成（无需重新运行 Cell）
if MANUAL_UPLOAD and WAIT_FOR_UPLOAD:
    start = time.time()
    while True:
        if ZIP_PATH and os.path.exists(ZIP_PATH):
            break
        if os.path.exists(UPLOAD_DIR):
            zips = glob.glob(os.path.join(UPLOAD_DIR, '*.zip'))
            if zips:
                break
        if uploader is not None and getattr(uploader, 'value', None):
            break
        if time.time() - start > WAIT_TIMEOUT_SEC:
            print('⚠️ 等待上传超时，请手动重新运行本 Cell 或设置 ZIP_PATH')
            break
        time.sleep(1)


def pick_zip_file(zip_path=None):
    if zip_path:
        if os.path.exists(zip_path) and zip_path.lower().endswith('.zip'):
            return zip_path
        raise FileNotFoundError(f'ZIP_PATH 不存在或不是 zip: {zip_path}')

    candidates = []
    for root in ['/kaggle/input', '/kaggle/working']:
        if os.path.exists(root):
            candidates.extend(glob.glob(os.path.join(root, '**', '*.zip'), recursive=True))

    if not candidates:
        raise FileNotFoundError(
            '未找到 .zip 文件。请先把 Insight 导出的 zip 上传到 Kaggle（/kaggle/input 或 /kaggle/working），或在本 Cell 设置 ZIP_PATH。'
        )

    insight_candidates = [p for p in candidates if 'insight' in os.path.basename(p).lower()]
    pool = insight_candidates if insight_candidates else candidates
    return sorted(pool, key=os.path.getmtime, reverse=True)[0]

zip_file = pick_zip_file(ZIP_PATH)
print(f'✅ 使用 ZIP: {zip_file}')

zip_stem = Path(zip_file).stem
extract_path = os.path.join(DATASETS_ROOT if 'DATASETS_ROOT' in globals() else '/kaggle/working/InsightYOLO/datasets', zip_stem)

has_existing = os.path.exists(extract_path) and os.path.exists(os.path.join(extract_path, 'data.yaml'))
if has_existing and not force_reextract:
    print(f'♻️ 发现已解压数据集，直接复用: {extract_path}')
else:
    if os.path.exists(extract_path):
        shutil.rmtree(extract_path)
    os.makedirs(extract_path, exist_ok=True)
    with zipfile.ZipFile(zip_file, 'r') as zf:
        zf.extractall(extract_path)
    print(f'✅ 已解压到: {extract_path}')

yaml_candidates = glob.glob(os.path.join(extract_path, '**', 'data.yaml'), recursive=True)
if not yaml_candidates:
    raise FileNotFoundError('解压后未找到 data.yaml，请确认 ZIP 是否正确')

data_yaml_found = sorted(yaml_candidates, key=len)[0]
dataset_path = os.path.dirname(data_yaml_found)
dataset_path, data_yaml, train_cfg, model_family, model_size, model_name = load_train_context(dataset_path)

save_state(
    last_dataset_path=dataset_path,
    last_source='zip_upload',
    zip_name=os.path.basename(zip_file),
    zip_path=zip_file,
    train_cfg=train_cfg,
    model_family=model_family,
    model_size=model_size,
    model_name=model_name
)


### 2.2 Kaggle 数据集目录（手动指定路径）


In [ ]:
# 手动指定已存在的数据集目录（目录下需有 data.yaml）
# 可填 /kaggle/input/... 或 /kaggle/working/InsightYOLO/datasets/...
dataset_path = '/kaggle/input/your_yolo_dataset'

if not os.path.exists(os.path.join(dataset_path, 'data.yaml')):
    raise FileNotFoundError(f'未找到 data.yaml，请检查路径: {dataset_path}')

dataset_path, data_yaml, train_cfg, model_family, model_size, model_name = load_train_context(dataset_path)
save_state(
    last_dataset_path=dataset_path,
    last_source='kaggle_manual',
    train_cfg=train_cfg,
    model_family=model_family,
    model_size=model_size,
    model_name=model_name
)


### 2.3 Roboflow 数据集（自动持久化到 /kaggle/working）


In [ ]:
from roboflow import Roboflow

WORKSPACE = 'YOUR_WORKSPACE'
PROJECT = 'YOUR_PROJECT'
VERSION = 1

rf = Roboflow(api_key='YOUR_API_KEY')
project = rf.workspace(WORKSPACE).project(PROJECT)
version = project.version(VERSION)
dataset = version.download('yolov8')

download_path = dataset.location
safe_name = f'roboflow_{WORKSPACE}_{PROJECT}_v{VERSION}'.replace('/', '_').replace(' ', '_')
persist_path = os.path.join(DATASETS_ROOT, safe_name)

if not os.path.exists(os.path.join(persist_path, 'data.yaml')):
    if os.path.exists(persist_path):
        shutil.rmtree(persist_path)
    shutil.copytree(download_path, persist_path)
    print(f'✅ Roboflow 数据集已持久化: {persist_path}')
else:
    print(f'♻️ 复用已持久化 Roboflow 数据集: {persist_path}')

fallback_cfg = {
    'yolo_version': 'yolov8',
    'model_size': 'n',
    'data_yaml': 'data.yaml'
}

dataset_path, data_yaml, train_cfg, model_family, model_size, model_name = load_train_context(persist_path, fallback_cfg)
save_state(
    last_dataset_path=dataset_path,
    last_source='roboflow',
    train_cfg=train_cfg,
    model_family=model_family,
    model_size=model_size,
    model_name=model_name
)


## 3) 开始训练（断点续训优先，参数来自软件导出配置）


In [ ]:
from ultralytics import YOLO
import os
import time
import yaml

# 掉线恢复：若当前会话没有 dataset_path，尝试从状态文件恢复
if 'dataset_path' not in globals() or not os.path.exists(str(globals().get('dataset_path', ''))):
    state = load_state()
    recovered_path = state.get('last_dataset_path')
    if recovered_path and os.path.exists(os.path.join(recovered_path, 'data.yaml')):
        print(f'♻️ 从状态恢复数据集: {recovered_path}')
        dataset_path, data_yaml, train_cfg, model_family, model_size, model_name = load_train_context(recovered_path)
    else:
        raise RuntimeError('未找到可用 dataset_path，请先执行 2.0 或 2.1/2.2/2.3')

def build_resolved_data_yaml(dataset_path, data_yaml):
    with open(data_yaml, 'r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f) or {}

    base = cfg.get('path')
    if not base or str(base).strip() in ('.', './'):
        base_dir = dataset_path
    else:
        base = str(base).strip()
        base_dir = base if os.path.isabs(base) else os.path.normpath(os.path.join(dataset_path, base))

    for key in ('train', 'val', 'test'):
        if key in cfg and cfg.get(key):
            p = str(cfg[key]).strip()
            if not os.path.isabs(p):
                if p.startswith('./'):
                    p = p[2:]
                p = os.path.normpath(os.path.join(base_dir, p))
            cfg[key] = p.replace('\\', '/')

    cfg.pop('path', None)
    fixed_yaml = os.path.join(dataset_path, 'data.resolved.yaml')
    with open(fixed_yaml, 'w', encoding='utf-8') as f:
        yaml.safe_dump(cfg, f, allow_unicode=True, sort_keys=False)

    for key in ('train', 'val'):
        p = cfg.get(key)
        if not p or not os.path.exists(p):
            raise FileNotFoundError(f'{key} 路径不存在: {p}')

    return fixed_yaml

# 规范化 device 配置（支持中文逗号）
raw_device = train_cfg.get('device', 0)
if isinstance(raw_device, str):
    device_str = _normalize_device_str(raw_device)
else:
    device_str = str(raw_device)

device_list = _parse_device_list(device_str) if device_str is not None else []

# 若用户配置多卡但实际不足，自动回退到可用范围
try:
    import torch
    gpu_count = torch.cuda.device_count()
    if device_list:
        valid = [d for d in device_list if 0 <= d < gpu_count]
        if len(valid) != len(device_list):
            print(f'⚠️ 你设置的 GPU Index={device_list}，但当前可用 GPU 数量={gpu_count}。')
            if gpu_count > 0:
                valid = list(range(gpu_count))
                print(f'↪️ 自动回退为可用设备: {valid}')
            else:
                valid = []
        device_list = valid
except Exception as e:
    print('⚠️ 无法校验 GPU 数量:', e)

# device 参数传给 Ultralytics
if device_list:
    device_arg = ','.join(str(i) for i in device_list)
else:
    device_arg = device_str if device_str is not None else 0

def start_new_training(model_name, fixed_yaml, train_cfg, run_name, runs_project):
    model = YOLO(model_name)
    return model.train(
        data=fixed_yaml,
        epochs=int(train_cfg.get('epochs', 100)),
        imgsz=int(train_cfg.get('img_size', 640)),
        batch=int(train_cfg.get('batch_size', 16)),
        name=run_name,
        project=runs_project,
        exist_ok=True,
        save=True,
        save_period=1,
        patience=int(train_cfg.get('patience', 50)),
        workers=int(train_cfg.get('workers', 8)),
        device=device_arg
    )

fixed_yaml = build_resolved_data_yaml(dataset_path, data_yaml)
print(f'✅ 使用训练配置: {fixed_yaml}')

if 'RUNS_PROJECT' not in globals():
    RUNS_PROJECT = '/kaggle/working/InsightYOLO/runs'
    os.makedirs(RUNS_PROJECT, exist_ok=True)

state = load_state()
run_name, resume_ckpt, resume_available, resume_reason, dataset_tag = resolve_resume_checkpoint(
    dataset_path=dataset_path,
    model_family=model_family,
    model_size=model_size,
    runs_project=RUNS_PROJECT,
    state=state
)

print(f'🧭 当前 run_name: {run_name}')
print(f'🧭 数据集标签: {dataset_tag}')

resume_ckpt_used = None
resume_error = None

if resume_available:
    training_mode = 'resume'
    resume_ckpt_used = resume_ckpt
    print(f'🔁 检测到断点，继续训练: {resume_ckpt}')
    print(f'   来源: {resume_reason}')
    try:
        model = YOLO(resume_ckpt)
        results = model.train(resume=True)
    except Exception as e:
        resume_error = str(e)
        fallback_run_name = f"{run_name}_restart_{time.strftime('%Y%m%d_%H%M%S')}"
        print(f'⚠️ 续训失败，自动回退新训练: {resume_error}')
        print(f'🚀 回退 run_name: {fallback_run_name}')
        run_name = fallback_run_name
        training_mode = 'new_after_resume_failed'
        results = start_new_training(model_name, fixed_yaml, train_cfg, run_name, RUNS_PROJECT)
else:
    training_mode = 'new'
    print('🚀 未检测到断点，开始新训练')
    results = start_new_training(model_name, fixed_yaml, train_cfg, run_name, RUNS_PROJECT)

resume_ckpt = os.path.join(RUNS_PROJECT, run_name, 'weights', 'last.pt')
resume_available_after_train = os.path.exists(resume_ckpt)

save_state(
    last_dataset_path=dataset_path,
    last_source='train_cell',
    train_cfg=train_cfg,
    model_family=model_family,
    model_size=model_size,
    model_name=model_name,
    dataset_tag=dataset_tag,
    run_name=run_name,
    runs_project=RUNS_PROJECT,
    resume_reason=resume_reason,
    resume_ckpt=resume_ckpt,
    resume_ckpt_used=resume_ckpt_used,
    resume_available=resume_available_after_train,
    resume_error=resume_error,
    training_mode=training_mode,
    results_dir=str(results.save_dir)
)

print('🎉 训练完成！')
print(f'📁 结果目录: {results.save_dir}')
if training_mode == 'resume':
    print('✅ 本次为断点续训。')
elif training_mode == 'new_after_resume_failed':
    print('⚠️ 原断点不可用，已自动切换为新训练。')
else:
    print('✅ 本次为新训练。')


### 3.1 续训状态提示（可单独运行）


In [ ]:
import os, csv, time

state = load_state()
runs_root = globals().get('RUNS_PROJECT') or state.get('runs_project') or '/kaggle/working/InsightYOLO/runs'
run_name_local = globals().get('run_name') or state.get('run_name')

if not run_name_local and 'model_family' in globals() and 'model_size' in globals():
    run_name_local = f'{model_family}{model_size}_train'

if not run_name_local:
    raise RuntimeError('无法确定 run_name，请先运行训练 Cell 或 2.0 自动恢复')

run_dir = os.path.join(runs_root, run_name_local)
last_pt = os.path.join(run_dir, 'weights', 'last.pt')
results_csv = os.path.join(run_dir, 'results.csv')

mode = globals().get('training_mode', None) or state.get('training_mode', None)
if mode == 'resume':
    print('✅ 本次训练模式：续训（resume）')
elif mode == 'new':
    print('✅ 本次训练模式：新训练（new）')
elif mode == 'new_after_resume_failed':
    print('⚠️ 本次训练模式：续训失败后回退新训练（new_after_resume_failed）')
else:
    print('⚠️ 当前会话未记录 training_mode（可能是重连后只运行了本 Cell）')

if state.get('resume_reason'):
    print(f"resume source: {state.get('resume_reason')}")
if state.get('resume_ckpt_used'):
    print(f"resume ckpt used: {state.get('resume_ckpt_used')}")
if state.get('resume_error'):
    print(f"resume error: {state.get('resume_error')}")

print(f'run_dir: {run_dir}')
print(f'last.pt exists: {os.path.exists(last_pt)}')
if os.path.exists(last_pt):
    print('last.pt mtime:', time.ctime(os.path.getmtime(last_pt)))

if os.path.exists(results_csv):
    with open(results_csv, 'r', encoding='utf-8') as f:
        rows = list(csv.reader(f))
    logged_epochs = max(0, len(rows) - 1)
    print(f'logged epochs: {logged_epochs}')
    if len(rows) > 1:
        print('last epoch row:', rows[-1][:6])
else:
    print('results.csv not found yet')


## 4) 查看训练结果


In [ ]:
from IPython.display import Image, display
import os, glob, csv

state = load_state()
runs_root = globals().get('RUNS_PROJECT') or state.get('runs_project') or '/kaggle/working/InsightYOLO/runs'
run_name_local = globals().get('run_name') or state.get('run_name')
if not run_name_local and 'model_family' in globals() and 'model_size' in globals():
    run_name_local = f'{model_family}{model_size}_train'

if 'results' in globals():
    save_dir = str(results.save_dir)
else:
    if not run_name_local:
        raise RuntimeError('无法定位结果目录，请先运行训练 Cell 或 3.1 状态提示 Cell')
    save_dir = os.path.join(runs_root, run_name_local)

print(f'📊 训练结果目录：{save_dir}\n')

results_img = os.path.join(save_dir, 'results.png')
if os.path.exists(results_img):
    print('📈 训练曲线：')
    display(Image(results_img, width=900))

val_imgs = glob.glob(os.path.join(save_dir, 'val_batch*.jpg'))
if val_imgs:
    print('\n🖼️  验证集检测示例：')
    display(Image(val_imgs[0], width=900))

results_csv = os.path.join(save_dir, 'results.csv')
if os.path.exists(results_csv):
    with open(results_csv, 'r', encoding='utf-8') as f:
        rows = list(csv.reader(f))
    if len(rows) > 1:
        header, last = rows[0], rows[-1]
        print('\n📋 关键指标（最后一轮）：')
        wanted = ['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/precision(B)', 'metrics/recall(B)']
        idx_map = {k: header.index(k) for k in wanted if k in header}
        for k in wanted:
            if k in idx_map:
                print(f'   {k}: {last[idx_map[k]]}')


## 5) 导出 ONNX


In [ ]:
from ultralytics import YOLO
import shutil
import yaml

state = load_state()
runs_root = globals().get('RUNS_PROJECT') or state.get('runs_project') or '/kaggle/working/InsightYOLO/runs'
run_name_local = globals().get('run_name') or state.get('run_name')
if not run_name_local and 'model_family' in globals() and 'model_size' in globals():
    run_name_local = f'{model_family}{model_size}_train'

if 'results' in globals():
    best_pt = os.path.join(str(results.save_dir), 'weights', 'best.pt')
else:
    if not run_name_local:
        raise RuntimeError('无法确定 run_name，请先执行训练 Cell 或恢复状态')
    best_pt = os.path.join(runs_root, run_name_local, 'weights', 'best.pt')

if not os.path.exists(best_pt):
    raise FileNotFoundError(f'best.pt 不存在: {best_pt}')

if 'train_cfg' not in globals():
    ds = state.get('last_dataset_path')
    if ds and os.path.exists(os.path.join(ds, 'data.yaml')):
        dataset_path, data_yaml, train_cfg, model_family, model_size, model_name = load_train_context(ds)
    else:
        raise RuntimeError('train_cfg 丢失且无法从状态恢复，请先运行 2.0 或训练 Cell')

model = YOLO(best_pt)
print(f'加载模型: {best_pt}')

imgsz = int(train_cfg.get('img_size', 640))
print('正在导出 ONNX...')
export_path = model.export(
    format='onnx',
    imgsz=imgsz,
    simplify=True,
    opset=12,
    dynamic=False
)

output_name = f'detector_{model_family}{model_size}.onnx'
export_root = EXPORTS_ROOT if 'EXPORTS_ROOT' in globals() else '/kaggle/working/InsightYOLO/exports'
os.makedirs(export_root, exist_ok=True)
final_onnx = os.path.join(export_root, output_name)
shutil.copy(export_path, final_onnx)

file_size = os.path.getsize(final_onnx) / 1024 / 1024
print('\n✅ ONNX 导出成功')
print(f'文件路径: {final_onnx}')
print(f'文件大小: {file_size:.2f} MB')
print(f'输入尺寸: {imgsz} x {imgsz}')

with open(data_yaml, 'r', encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

print(f'类别数量: {data_cfg.get("nc")}')
print(f'类别列表: {data_cfg.get("names")}')

save_state(last_onnx=final_onnx, last_onnx_size_mb=round(file_size, 2))


## 6) 准备下载 ONNX（Kaggle Files）


In [ ]:
import os, shutil

if 'final_onnx' not in globals() or not os.path.exists(str(globals().get('final_onnx', ''))):
    state = load_state()
    if state.get('last_onnx') and os.path.exists(state['last_onnx']):
        final_onnx = state['last_onnx']
        output_name = os.path.basename(final_onnx)
        print(f'♻️ 从状态恢复 ONNX 路径: {final_onnx}')
    else:
        raise FileNotFoundError('未找到可导出的 ONNX，请先执行导出 ONNX Cell')

kaggle_download_path = os.path.join('/kaggle/working', os.path.basename(final_onnx))
if os.path.abspath(final_onnx) != os.path.abspath(kaggle_download_path):
    shutil.copy(final_onnx, kaggle_download_path)

print('✅ ONNX 已准备到 Kaggle 工作目录')
print(f'文件路径: {kaggle_download_path}')
print('在 Kaggle 右侧 Files 面板中可直接下载该文件。')

save_state(last_onnx_download_ready=kaggle_download_path)


## 7) （可选）快速推理验证


In [ ]:
import os, glob, random
import cv2
from IPython.display import Image as IPImage, display
from ultralytics import YOLO

state = load_state()
runs_root = globals().get('RUNS_PROJECT') or state.get('runs_project') or '/kaggle/working/InsightYOLO/runs'
run_name_local = globals().get('run_name') or state.get('run_name')
if not run_name_local and 'model_family' in globals() and 'model_size' in globals():
    run_name_local = f'{model_family}{model_size}_train'

if 'dataset_path' not in globals() or not os.path.exists(str(globals().get('dataset_path', ''))):
    ds = state.get('last_dataset_path')
    if ds and os.path.exists(os.path.join(ds, 'data.yaml')):
        dataset_path, data_yaml, train_cfg, model_family, model_size, model_name = load_train_context(ds)
    else:
        raise RuntimeError('无法恢复 dataset_path，请先运行 2.0 自动恢复或数据源 Cell')

if 'results' in globals():
    best_pt = os.path.join(str(results.save_dir), 'weights', 'best.pt')
else:
    if not run_name_local:
        raise RuntimeError('无法确定 run_name，请先运行训练 Cell 或 3.1 状态提示 Cell')
    best_pt = os.path.join(runs_root, run_name_local, 'weights', 'best.pt')

if not os.path.exists(best_pt):
    raise FileNotFoundError(f'best.pt 不存在: {best_pt}')

candidate_dirs = [
    os.path.join(dataset_path, 'test', 'images'),
    os.path.join(dataset_path, 'valid', 'images'),
    os.path.join(dataset_path, 'images', 'val'),
    os.path.join(dataset_path, 'images', 'train'),
]

img_dir = next((d for d in candidate_dirs if os.path.exists(d)), None)
if not img_dir:
    print('❌ 未找到可用图像目录')
else:
    all_imgs = glob.glob(os.path.join(img_dir, '*.jpg')) + glob.glob(os.path.join(img_dir, '*.png'))
    if not all_imgs:
        print(f'❌ 在 {img_dir} 未找到图像')
    else:
        sample_img = random.choice(all_imgs)
        print(f'🖼️ 测试图像：{os.path.basename(sample_img)}')
        print(f'使用权重：{best_pt}')

        pred_root = os.path.join('/kaggle/working/InsightYOLO', 'predictions')
        os.makedirs(pred_root, exist_ok=True)

        infer_model = YOLO(best_pt)
        pred_results = infer_model.predict(
            source=sample_img,
            conf=0.25,
            save=True,
            project=pred_root,
            name='test',
            exist_ok=True
        )

        pred_img = glob.glob(os.path.join(pred_root, 'test', '*.jpg')) + glob.glob(os.path.join(pred_root, 'test', '*.png'))
        if pred_img:
            display(IPImage(pred_img[0], width=700))

        if 'data_cfg' not in globals():
            import yaml
            with open(data_yaml, 'r', encoding='utf-8') as f:
                data_cfg = yaml.safe_load(f)

        result = pred_results[0]
        print(f'\n检测到 {len(result.boxes)} 个目标：')
        for box in result.boxes:
            cls_name = data_cfg['names'][int(box.cls)]
            conf = float(box.conf)
            print(f'   - {cls_name}  置信度: {conf:.2%}')
